# Tree Segmentation - Marker-Controlled Watershed

Same CHM, smoothing, treetops (`lmf`) and over-segmentation diagnostics as `tree_segmentation_dalponte.ipynb`, but crowns are delineated with a **marker-controlled watershed** (`ForestTools::mcws`). Dalponte is also run so the two can be compared directly.

## How the watershed works
Flip the CHM upside down: each tree crown becomes a basin and the valleys between crowns become ridges. Starting from each treetop (the **marker**), the basin floods outward until it meets the flood from a neighbouring marker; the meeting line is the crown boundary. Cells below `min_height` are masked out, so ground between tree rows is not assigned to any tree.

## Should you use it? Yes as a comparison, probably not as a replacement

**For an orchard it is a good fit:** crowns are isolated or only slightly touching, roughly round, evenly spaced, and surrounded by low ground. Boundaries between touching crowns follow the actual valley in the CHM instead of a height-ratio rule, and there are no growth thresholds to tune.

**Caveats**
- It is only as good as the markers. The same `lmf` treetops go in, so watershed will **not fix over-segmentation** - a double top gives two basins.
- Every above-threshold cell is assigned to *some* tree, so crowns are larger (include the soft edge) than Dalponte's, which stops at `th_cr`. Crown area/diameter are biased high unless `min_height` is well above the ground/weed layer.
- Basins can leak into neighbouring weeds or cover crop if `min_height` is too low.
- For your downstream use (per-tree height, count per footprint), heights come from the treetops, so **the choice between watershed and Dalponte barely changes the biomass inputs**; it matters for crown area, crown diameter, and the over-segmentation checks.

Refs: Hyyppa et al. 2001 and Chen et al. 2006 (marker-controlled watershed on a CHM); [fruit-tree crown extraction with local maximum + marker-controlled watershed](https://www.sciencedirect.com/science/article/pii/S2666719322000644).

## 1. Packages

In [ ]:
# install.packages(c("terra", "sf", "lidR", "dplyr", "ForestTools", "igraph"))   # igraph is needed by mcws
suppressPackageStartupMessages({ library(terra); library(sf); library(lidR); library(dplyr); library(ForestTools) })
source("../scripts/segmentation_helpers.R")   # crown stats + over-segmentation diagnostics + plots

## 2. Configuration

In [ ]:
cfg <- list(
  res        = "0.5m",   # CHM resolution to use: "0.5m" (recommended) or "0.25m" (sensitivity check)
  smooth_k   = 3,        # median smoothing kernel (cells). 3x3 @0.5 m = 1.5 m
  lmf_ws     = 3.0,      # local-maximum window (m): ~ half a typical crown diameter
  min_height = 2.0,      # ignore anything below this (m): ground, weeds, cover crop
  # Dalponte settings (only used for the comparison in section 7)
  th_tree    = 2.0,      # Dalponte: min height of any crown cell (m)
  th_seed    = 0.45,     # Dalponte: cell must be > th_seed x seed height
  th_cr      = 0.55,     # Dalponte: cell must be > th_cr x mean crown height
  max_cr_m   = 10,       # Dalponte: max crown extent (m) - converted to pixels below
  # over-segmentation diagnostics
  os_min_prom = 1.0,     # shorter top must rise >= this (m) above the saddle, else the pair is suspect
  os_max_dist = 4.0,     # ...only checked for tops closer than this (m)
  os_min_area = 3.0,     # crowns smaller than this (m2) are flagged as slivers
  # ground-truth comparison
  match_max_dist = 2.5,  # max distance (m) between a surveyed tree and its segmented treetop
  outlier_m      = 5     # matches with |height error| >= this are also reported separately
)

## 3. Load CHM, smooth, detect treetops

In [ ]:
# Paths are relative to the notebooks/ folder
chm_path <- sprintf("../../Rasters/bart_chm_%s_clipped.tif", cfg$res)
ground_path <- "../../Rasters/BR_Ground_Trees.gpkg"
out_dir  <- file.path("../outputs/Bartleson Tree Segmentation", paste0("watershed_", cfg$res))
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

chm <- rast(chm_path)
res_m <- res(chm)[1]
cat("CHM:", basename(chm_path), "| resolution:", res_m, "m\n")

# Smooth + detect treetops (shared by both segmentation methods)
chm_smooth <- focal(chm, w = matrix(1, cfg$smooth_k, cfg$smooth_k), fun = median, na.rm = TRUE)
ttops <- locate_trees(chm_smooth, lmf(ws = cfg$lmf_ws, hmin = cfg$min_height))
ttops$height <- st_coordinates(ttops)[, "Z"]
cat("Treetops detected:", nrow(ttops), "\n")

## 4. Marker-controlled watershed

In [ ]:
crowns <- mcws(treetops = ttops, CHM = chm_smooth, minHeight = cfg$min_height)   # SpatRaster of treeIDs
names(crowns) <- "treeID"

## 5. Over-segmentation diagnostics

In [ ]:
polys <- crown_polygons(crowns, chm_smooth, ttops)
pairs <- adjacent_pairs(crowns, chm_smooth, ttops)
fl    <- flag_overseg(polys, pairs, cfg$os_min_prom, cfg$os_max_dist, cfg$os_min_area)
polys <- fl$polys; suspect_pairs <- fl$suspect_pairs

cat("Crowns:", nrow(polys), "| median crown diameter:", round(median(polys$diam_m), 2), "m\n")
print(table(polys$flag))
cat(sprintf("Suspect over-segmented pairs: %d  (%.1f%% of crowns involved)\n",
            nrow(suspect_pairs), 100 * mean(polys$in_suspect_pair)))

## 6. Visualization

In [ ]:
# Whole-scene map: crowns coloured by over-segmentation flag
options(repr.plot.width = 14, repr.plot.height = 12)
flag_cols <- c("ok" = "#cccccc", "small crown" = "#f4a300", "suspect pair" = "#e03131", "suspect pair + small" = "#7b0000")
plot(chm_smooth, col = gray.colors(50, 0.3, 1), legend = FALSE, main = "All crowns - flagged for possible over-segmentation")
plot(st_geometry(polys), col = adjustcolor(flag_cols[polys$flag], 0.7), border = NA, add = TRUE)
legend("topright", legend = names(flag_cols), fill = adjustcolor(flag_cols, 0.7), bg = "white")

In [ ]:
options(repr.plot.width = 24, repr.plot.height = 8)

# The 4 most over-segmented 30 m blocks in the scene: CHM+tops | crowns | flags
blk <- suspect_pairs |>
  mutate(bx = floor((xa + xb) / 2 / 30), by = floor((ya + yb) / 2 / 30)) |>
  count(bx, by, sort = TRUE) |> head(4)
for (k in seq_len(nrow(blk))) {
  w <- ext(blk$bx[k] * 30, (blk$bx[k] + 1) * 30, blk$by[k] * 30, (blk$by[k] + 1) * 30)
  plot_overseg_window(chm_smooth, polys, ttops, suspect_pairs, w,
                      title = sprintf("Block %d (%d suspect pairs)", k, blk$n[k]))
}

In [ ]:
# The most likely over-segmentations in the whole scene (lowest prominence first)
worst <- suspect_pairs |> arrange(prominence) |> head(6)
for (k in seq_len(nrow(worst))) {
  xm <- (worst$xa[k] + worst$xb[k]) / 2; ym <- (worst$ya[k] + worst$yb[k]) / 2
  plot_overseg_window(chm_smooth, polys, ttops, suspect_pairs, ext(xm - 6, xm + 6, ym - 6, ym + 6),
                      title = sprintf("Pair %d-%d: prominence %.2f m, tops %.1f m apart",
                                      worst$a[k], worst$b[k], worst$prominence[k], worst$dist_m[k]))
}

## 7. Watershed vs Dalponte (same treetops)

In [ ]:
crowns_dal <- dalponte2016(chm_smooth, ttops, th_tree = cfg$th_tree, th_seed = cfg$th_seed,
                           th_cr = cfg$th_cr, max_cr = ceiling(cfg$max_cr_m / res_m))()
polys_dal <- crown_polygons(crowns_dal, chm_smooth, ttops)

cmp <- inner_join(
  st_drop_geometry(polys)     |> select(treeID, area_ws = area_m2),
  st_drop_geometry(polys_dal) |> select(treeID, area_dal = area_m2), by = "treeID")

cat(sprintf("Median crown area  watershed: %.1f m2 | Dalponte: %.1f m2\n", median(cmp$area_ws), median(cmp$area_dal)))

options(repr.plot.width = 14, repr.plot.height = 6)
par(mfrow = c(1, 2))
plot(cmp$area_dal, cmp$area_ws, pch = 16, col = adjustcolor("black", 0.25),
     xlab = "Dalponte crown area (m2)", ylab = "Watershed crown area (m2)", main = "Per-tree crown area")
abline(0, 1, col = "red", lwd = 2)
boxplot(list(Dalponte = polys_dal$diam_m, Watershed = polys$diam_m), ylab = "Crown diameter (m)", main = "Crown diameter")
par(mfrow = c(1, 1))

# Side-by-side on the most over-segmented block
w  <- ext(blk$bx[1] * 30, (blk$bx[1] + 1) * 30, blk$by[1] * 30, (blk$by[1] + 1) * 30)   # most over-segmented block
bb <- st_bbox(c(xmin = xmin(w), xmax = xmax(w), ymin = ymin(w), ymax = ymax(w)), crs = st_crs(ttops))
options(repr.plot.width = 16, repr.plot.height = 8)
par(mfrow = c(1, 2))
for (nm in c("Dalponte", "Watershed")) {
  p <- if (nm == "Dalponte") polys_dal else polys
  plot(crop(chm_smooth, w), col = gray.colors(50, 0.3, 1), legend = FALSE, main = nm)
  plot(st_geometry(suppressWarnings(st_crop(p, bb))), border = "red", lwd = 2, add = TRUE, col = NA)
  plot(st_geometry(ttops[st_intersects(ttops, st_as_sfc(bb), sparse = FALSE)[, 1], ]), add = TRUE, pch = 3, lwd = 2)
}
par(mfrow = c(1, 1))

## 8. Compare to ground-measured heights

Same matching as the Dalponte notebook (heights come from the shared treetops, so only the raw-CHM-max column depends on the crowns).

In [ ]:
ground  <- read_ground_trees(ground_path, st_crs(chm))
matched <- match_ground_trees(ground, ttops, polys, chm, cfg$match_max_dist)

cat(sprintf("\nMatched %d of %d surveyed trees within %.1f m (median distance %.2f m)\n",
            nrow(matched), nrow(ground), cfg$match_max_dist, median(matched$dist_m)))

metrics <- rbind(`smoothed treetop height` = height_metrics(matched$seg_height_m, matched$gt_height_m),
                 `raw CHM max in crown`    = height_metrics(matched$raw_max_m,    matched$gt_height_m))
cat("All matches:\n"); print(round(metrics, 3))

# Same, dropping gross mismatches (likely a wrong match or a bad field height, not segmentation error)
ok <- abs(matched$err_seg) < cfg$outlier_m
cat(sprintf("\nExcluding %d match(es) with |error| >= %.0f m:\n", sum(!ok), cfg$outlier_m))
print(round(rbind(`smoothed treetop height` = height_metrics(matched$seg_height_m[ok], matched$gt_height_m[ok]),
                  `raw CHM max in crown`    = height_metrics(matched$raw_max_m[ok],    matched$gt_height_m[ok])), 3))
cat("\nNote: R2 is weak evidence here - surveyed heights span only ~2.4-3.5 m.\n")

In [ ]:
options(repr.plot.width = 18, repr.plot.height = 6)
par(mfrow = c(1, 3))
lim <- range(c(matched$gt_height_m, matched$seg_height_m, matched$raw_max_m), na.rm = TRUE)
for (v in c("seg_height_m", "raw_max_m")) {
  plot(matched$gt_height_m, matched[[v]], xlim = lim, ylim = lim, pch = 16, col = adjustcolor("steelblue", 0.7),
       xlab = "Ground-measured height (m)", ylab = "Segmented height (m)",
       main = if (v == "seg_height_m") "Smoothed treetop height" else "Raw CHM max in crown")
  abline(0, 1, col = "red", lwd = 2)
}
hist(matched$err_seg, breaks = 20, col = "grey70", main = "Segmented - ground (smoothed treetop)", xlab = "error (m)")
abline(v = 0, col = "red", lwd = 2)
par(mfrow = c(1, 1))

# Where the matches are: surveyed tree -> segmented treetop link on the CHM
options(repr.plot.width = 14, repr.plot.height = 12)
plot(chm_smooth, col = gray.colors(50, 0.3, 1), legend = FALSE, main = "Surveyed trees (green) matched to segmented treetops (black)",
     ext = ext(vect(st_buffer(st_as_sfc(st_bbox(ground)), 10))))
plot(st_geometry(polys), border = adjustcolor("black", 0.4), col = NA, add = TRUE)
plot(st_geometry(ground), add = TRUE, pch = 1, col = "darkgreen", cex = 1.2, lwd = 2)
plot(st_geometry(ttops), add = TRUE, pch = 3, col = "black")
segments(matched$xg, matched$yg, matched$xt, matched$yt, col = "red", lwd = 2)

# Largest height errors
matched |> arrange(desc(abs(err_seg))) |> select(label, dist_m, gt_height_m, seg_height_m, raw_max_m, err_seg) |> head(10)

## 9. Distributions

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)
par(mfrow = c(1, 2))
hist(polys$diam_m, breaks = 40, col = "grey70", main = "Crown equivalent diameter", xlab = "m")
hist(pairs$prominence[pairs$dist_m < cfg$os_max_dist], breaks = 40, col = "grey70",
     main = paste0("Prominence of touching neighbours (<", cfg$os_max_dist, " m apart)"),
     xlab = "shorter top above saddle (m)")
abline(v = cfg$os_min_prom, col = "red", lwd = 2)
par(mfrow = c(1, 1))

## 10. Export

In [ ]:
tree_data <- data.frame(treeID = ttops$treeID, x = st_coordinates(ttops)[, "X"],
                        y = st_coordinates(ttops)[, "Y"], height_m = ttops$height) |>
  left_join(st_drop_geometry(polys) |> select(treeID, crown_area_m2 = area_m2, crown_diam_m = diam_m, flag), by = "treeID")
write.csv(tree_data, file.path(out_dir, "tree_heights.csv"), row.names = FALSE)
st_write(polys, file.path(out_dir, "crowns.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(ttops, file.path(out_dir, "treetops.gpkg"), delete_dsn = TRUE, quiet = TRUE)
cat("Saved to", normalizePath(out_dir), "\n")